In [4]:
import numpy as np
from scipy.integrate import quad
from scipy.integrate import odeint

from scipy.special import sph_harm
from scipy.special import gamma
from scipy.optimize import fsolve


G = 6.67e-11 # gravitational constant
AU = 1.49e11 # [m], astronomical unit
a = 0.723*AU # [m], Venus' semi-major axis
R_V = 6.0518e6 # [m], radius of Venus
M_V = 4.8685e24 # [kg], mass of Venus
M_sun = 1.989e30 # [kg], mass of Sun
Ifactor = 0.336 # moment of inertia factor for Venus
C = Ifactor*M_V*R_V**2 # moment of inertia Venus on spin axis
deltEd = 1.3e-5 # departure from dynamical ellipticity, pg. 3
AminBoverC = 2.16e-6 # axially asymmetric distribution of mass, pg. 3


## for gravitational tide ##

k2 = 0.295 # 2nd order love number 
n = 2*np.pi/(224.701*60*60*24)# mean motion, 2pi/T Venus

kf = 0.2 # fluid love number (idk??)

sec_per_year = 60*60*24*365
delt_t_g = 638 # s for Earth
Kg = (G*M_sun**2*R_V**5)/(C*a**6)


## for CMF ##

visc = 1e-2 # m2/s, CL, 2001
Rc = 3.2e6 # m, suggested core radius of Venus, Table 2 Correia 2006
cc = 0.084 # table 2, Correia 2006
Cc = 0.084*C
deltE_c = 4*deltEd
Ec = 4e-4#(Cc-Ac)/Cc, for now since I can't figure out how to calculate, weak coupling 
J2 = 4.5e-6 # (C - 0.5*(A+B))/(MR^2), Table 2, C2006

gamma_el = 0.75 # C2003, I, pg. 8, eq. 50
ed_ec = 1/4 # non-hydrostatic, 4/3 for hydrostatic 


## for thermal tide

g = 8.87 # m/s2
sigma_SB = 5.67e-8 # Stefan-Boltzmann constant [W m-2 K-4]

## constants used in Salazar & Wordsworth, 2023

#n = 2*np.pi/(224*60*60*24) # orbital mean motion [rad/s]
Cs_land = 1000000 # heat capacity of surface, [J kg-1]
cp = 1000 # specific heat of air, [J kg-1 K-1]
#rho_mean = 5510 # density of planet, [kg m-3]
rho_mean = 5204
D = 1.66 # diffusivity constant 
R = 188 # specific gas constant, [J kg-1 K-1]
Cd = 0.0034 # surface drag coefficient
chi = 0.17 # reference thermal coupling factor
bar = 1e5
## lumiosty 

Lo = 3.828e26
to = 4.57e9
alp = 0.45

ke = 0.25 # elastic love number
kf = 0.928 # fluid love number
tau_e = 1468 * 60 * 60 * 24 * 365 # elastic relaxation time, 1468 yrs
tau_tot = kf*tau_e/ke
alph = 0.3



Ka = (3*M_sun*R_V**3)/(5*C*rho_mean*a**3)

# Q = 50 # quality factor
# Qn = 50 # quality factor, TL


Omega_Venus = -2*np.pi/(244*60*60*24)
alpha_Venus = 0.77
ps_Venus = 92e5
S_Venus = 2624.3

kappa_lw = g/bar
kappa_sw = 0.3*g/bar



## make dF matrix [5x5] which calculates the spherical harmonic decompositions of S = max(cos(lam),0)
dF_matrix = np.zeros([3,5])
for n_sph in range(0,3):
    for m in range(-n_sph,n_sph+1):
        dF_matrix[n_sph,m+2] = quad(lambda x: np.maximum(np.cos(x),0)*np.real(sph_harm(m,n_sph,x,np.pi/2)),0, 2*np.pi)[0]


def alpha_fit(S, Omega, delta):
    Omega_crit = 2*np.pi/(40*60*60*24) # transition between cloudy and not cloudy
    alpha_slow = 0.5+0.17*(np.log2(S/1366)) # have min and max scale with stellar radiation 
    alpha_rapid = 0.35+0.08*(np.log2(S/1366))
    alpha = alpha_rapid + ((alpha_slow-alpha_rapid)/2)*(np.tanh((-Omega+Omega_crit)/delta) +1) # hyperbolic tangent
    return(alpha)

def Tbar(S, alpha, tau_sw, tau_lw):
    k = tau_sw/tau_lw 
    F_bar = S*(1-alpha)*np.exp(-tau_sw)/np.pi # average incident stellar radiation at surface (Wm-2)
    SLW = S*(1-alpha)/8*(1+D/k - (1+D/k)*np.exp(-k*tau_lw)) # surface downwelling longwave (Wm-2), Equation (15)
    T_bar = np.power((F_bar+SLW)/sigma_SB, 1/4) # average surface temperature (K)
    return(T_bar)

def wind_speed(S, alpha, tau_sw, tau_lw, ps): # Us [m/s], Equation (23)
    T_bar = Tbar(S, alpha, tau_sw, tau_lw)
    T_eq = (S*(1-alpha)/(4*sigma_SB))**(1/4) # equilibrium surface temperature (K)
    Us = np.power(R/Cd * np.maximum((T_bar - T_eq),0) * (S/2)*(1-alpha)*np.exp(-tau_sw)*(1-np.exp(-tau_lw))/ps,1/3) 
    return(Us)

def qo_wo(S, alpha, tau_sw, tau_lw, ps, m, l): # output q_o (defined in Equation 29) and w_o
    k = tau_sw/tau_lw
    F_bar = S*(1-alpha)*np.exp(-tau_sw)/np.pi # average incident stellar radiation at surface (Wm-2)
    SLW = S*(1-alpha)/8*(1+D/k - (1+D/k)*np.exp(-k*tau_lw)) # surface downwelling longwave (Wm-2), Equation (15)
    T_bar = np.power((F_bar+SLW)/sigma_SB, 1/4) # average surface temperature (K)
    delta_F = dF_matrix[2,2+2]
    T_eq = (S*(1-alpha)/(4*sigma_SB))**(1/4) # equilibrium surface temperature (K)
    Us = wind_speed(S, alpha, tau_sw, tau_lw, ps)
    circ_strength = Us/Uso # circulation strength, Equation (22)
    delt_p = chi*ps*circ_strength # Equation (23)
    Cs = cp*delt_p/g + Cs_land # heat capacity of surface, including atmosphere
    if ps == ps_Venus:
        w_o = 3.77e-7 # from Leconte
    else:
        w_o = 4*sigma_SB*T_bar**3/Cs # thermal equilibrium frequency
    qo = -(1/4)*delta_F*S*(1-alpha)*np.exp(-tau_sw)*delt_p/(F_bar + SLW)*np.sqrt(10/(3*np.pi))
    return(qo, w_o)

def torque_analytic_forcingfreq(S, alpha, tau_sw, tau_lw, ps, Omega, n, m, l): # q_tilde [Pa], Equation (29)
    qo, w_o = qo_wo(S, alpha, tau_sw, tau_lw, ps, m,l)
    sigma = (m*Omega-l*n)
    torque = -qo*sigma/w_o/(1+(sigma/w_o)**2) # Equation (29)
    return(torque)


def thermal_tide_torque(M, R, a,S, alpha, tau_sw, tau_lw, ps, Omega_list, n, m, l): # T_a, Equation (32)
    Ka = -3*M*R**3/((5*rho_mean*a**3))
    q_tilde = torque_analytic_forcingfreq(S, alpha, tau_sw, tau_lw, ps, Omega_list, n, m,l)
    return(-Ka*q_tilde)

def gravitational_tide_torque(M, R, a,Q,Qn, k2, Omega_list, n, rheo_model='Constant-Q'): # T_g, Equation (4)
    Kg = -G*M**2*R**5/a**6
    if rheo_model == 'Constant-Q':
        bg = k2/Q * np.sign(Omega_list-n) 
    elif rheo_model == 'CL2003':
        bg = k2/Q * np.sign(Omega_list-n)  * (1-(1-Q/Qn)**(np.absolute(2*(Omega_list-n))/n))
    elif rheo_model == 'Andrade':
        bg = Andrade(2*(Omega_list-n))
    else:
        print('Error: please enter valid rheology model')
        bg = np.nan
    return(Kg*bg)
def Andrade(sig):
    #sig = 2*(Omega_list-n)
    B_sig = 1 + np.absolute(sig*tau_tot)**(1-alph) * (tau_e/tau_tot)**(1-alph)*np.sin(alph*np.pi/2)*gamma(1+alph)
    A_sig = (sig*tau_tot)*(1 + np.maximum(np.absolute(sig*tau_tot),1e-3)**(-alph) * (tau_e/tau_tot)**(1-alph)*np.cos(alph*np.pi/2)*gamma(1+alph) )
    k2Q = (kf-ke) * B_sig*sig*tau_tot/(A_sig**2 + B_sig**2) 
    return(k2Q)
Uso = wind_speed(1137, 0.2, 0.00001, 1, 1*bar) # U_so [m/s]   
def modern_Venus_tuning(x, rheo_model):
    tau_lw = x[0]
    tau_sw = x[1]
    k = tau_sw/tau_lw
    S = S_Venus#2601.3
    alpha = 0.77
    D = 1.66
    F_bar = S*(1-alpha)*np.exp(-tau_sw)/np.pi # average incident stellar radiation at surface (Wm-2)
    SLW = S*(1-alpha)/8*(1+D/k - (1+D/k)*np.exp(-k*tau_lw)) # surface downwelling longwave (Wm-2), Equation (15)
    T_bar = np.power((F_bar+SLW)/sigma_SB, 1/4) # average surface temperature (K)
    Tbar = T_bar-737 # choose solution with T_bar=737 K
    rot_eq = gravitational_tide_torque(M_sun, R_V, a,Q,Qn, k2, Omega_Venus, n, rheo_model) + thermal_tide_torque(M_sun, R_V, a,S, alpha_Venus, tau_sw, tau_lw, ps_Venus, Omega_Venus, n,2,2)
    return(Tbar, rot_eq)